# NOVA 02 — MOD-04 Currency Detection (MobileNetV3-Small)
**10 classes**: notes 500/1000/2000/5000/10000 + coins 25/50/100/200/500 (incl. the new BEAC Type-2024 coin series).

**Prerequisite:** upload `cfa_currency_kaggle.zip` (built by `scrape_cfa_images.py` + `augment_cfa_dataset.py` — official BEAC scans, augmented; 1830 train / 120 val / 120 test) as a **private Kaggle dataset**, attach it here, and set CFA_DATA below.

The bootstrap dataset uses clean official scans + augmentation. Coin classes have leaky val/test (few sources) — treat coin test metrics as optimistic until team-collected photos are added.

In [ ]:
# ── NOVA bootstrap: clone repo + resolve HF token from Kaggle secret ──
# Before running: Add-ons > Secrets > attach a secret named HF_TOKEN
# (your HuggingFace write token). Settings > Accelerator > GPU T4/P100.
import os, sys, subprocess

REPO = 'https://github.com/BertinAm/nova-ml.git'
if not os.path.exists('/kaggle/working/nova-ml'):
    subprocess.run(['git', 'clone', REPO, '/kaggle/working/nova-ml'], check=True)
else:  # already cloned in this session — pull latest fixes
    subprocess.run(['git', '-C', '/kaggle/working/nova-ml', 'pull'], check=True)
os.chdir('/kaggle/working/nova-ml')
sys.path.insert(0, '/kaggle/working/nova-ml/scripts')

from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['NOVA_HF_USERNAME'] = 'unixio'

# GPU compatibility guard: Kaggle's PyTorch 2.10 image dropped sm_60 (P100).
# If you get a P100, switch Settings > Accelerator to 'GPU T4 x2'.
import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    assert cap >= (7, 0), (
        f'{name} (sm_{cap[0]}{cap[1]}) is NOT supported by this PyTorch build. '
        'Switch Settings > Accelerator to GPU T4 x2 and restart.')
    print(f'GPU OK: {name}')
else:
    raise RuntimeError('No GPU — set Settings > Accelerator to GPU T4 x2.')
print('Bootstrap OK — repo cloned, HF token loaded.')

In [ ]:
!pip install -q timm onnx2tf onnx

In [ ]:
# Resolve the attached dataset mount (search 3 levels — Kaggle may nest
# under /kaggle/input/datasets/<owner>/<slug>)
import glob, os
inputs = (glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*')
          + glob.glob('/kaggle/input/*/*/*'))
CFA_DATA = next(p for p in inputs
                if 'cfa' in p.split('/')[-1].lower() and os.path.isdir(p))
# If the zip extracted with a wrapper folder, descend into it
if not os.path.isdir(f'{CFA_DATA}/train'):
    CFA_DATA = next(p for p in glob.glob(f'{CFA_DATA}/*')
                    if os.path.isdir(f'{p}/train'))
print('CFA_DATA =', CFA_DATA)
!ls {CFA_DATA}/train

In [ ]:
# Two-phase: fine-tune EfficientNet-B4 teacher, distill into MobileNetV3-Small.
# Class count is auto-detected from the train/ folders (10).
!python scripts/train_currency_distillation.py --data-dir {CFA_DATA} \
    --teacher-epochs 20 --student-epochs 60 --batch-size 64

In [ ]:
# Held-out test evaluation at the 0.85 confidence gate (FR-04-03)
!python scripts/evaluate_models.py currency \
    --checkpoint /kaggle/working/checkpoints/currency_student_best.pth \
    --data-dir {CFA_DATA}/test

In [ ]:
# Convert to TFLite INT8 (calibrate on val images) + benchmark
!python scripts/convert_to_tflite.py \
    --checkpoint /kaggle/working/checkpoints/currency_student_best.pth \
    --arch mobilenetv3_small_100 --num-classes 10 --input-size 224 \
    --out /kaggle/working/exports/currency_detection_v1.tflite \
    --calib-dir {CFA_DATA}/val --benchmark

In [ ]:
# Publish to HuggingFace: unixio/nova-currency-detection
!python scripts/push_to_huggingface.py --module MOD-04 \
    --pytorch /kaggle/working/checkpoints/currency_student_best.pth \
    --tflite /kaggle/working/exports/currency_detection_v1.tflite \
    --labels labels/cfa_labels.txt \
    --eval-json /kaggle/working/evaluation/currency_test_results.json --version 1.0.0